# 共通コード
次のセルが実習で使う共通の関数などのコードになります。
次のセルを実行しておきましょう。

In [ ]:
# 共通コード
# このセルを実行してください
from PIL import Image
from IPython.display import display
from IPython.display import clear_output
import cv2
import numpy as np
from pathlib import Path

def convert_pil(im, bgr=True):
    if isinstance(im, np.ndarray):
        if bgr:
            im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        im = Image.fromarray(im)
    if isinstance(im, Image.Image):
        pass
    if isinstance(im, str):
        im = Image.open(im)
    if isinstance(im, Image.Image):
        pass
    else:
        raise ValueError("Invalid data type.")

    return im

# 画像を表示（PIL or numpy array(opencv) or file path）
def show_image(im, bgr=True, handle=None):
    im = convert_pil(im)

    if handle is None:
        display(im)
    else:
        handle.update(im)

# 複数の画像を表示（PIL or numpy array(opencv) or file path）
# titlesにいれた文字列をタイトルとして表示
# titlesの値がNoneの場合はfile pathの場合はファイル名を表示、それ以外（PIL or numpy arrya）の場合はタイトルを表示しない。   
def show_images(im_list, titles=[], bgr=True):    
    titles += [None] * len(im_list)
    for im, title in zip(im_list, titles):
        if title is None:
            if isinstance(im, str):
                title = Path(im).name
            else:
                title = ""
            
        print(title)
        show_image(im, bgr=bgr)

# 指定ディレクトリの画像一覧を取得する
def get_image_path_list(root_path, recursive=False, exts=None, pathlib=False):
    root_path = Path(root_path)
    
    if exts is None:
        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp'}
    else:
        exts = {e.lower() if e.startswith('.') else f'.{e.lower()}' for e in exts}
   
    if recursive == True:
        image_path_list = [f for f in root_path.rglob('*') if f.is_file() and f.suffix.lower() in exts]
    else:
        image_path_list = [f for f in root_path.iterdir() if f.is_file() and f.suffix.lower() in exts]

    if not pathlib:
        image_path_list = [str(p) for p in image_path_list]

    return image_path_list

# モデル学習（PatchCore）
Anomalibではいくつかの方法で学習やモデル評価を行うことができますが、この講習では学習・評価用のPythonコードを書いて実行してみます。
Anomalibではanomalib.engine.Engineというクラスが提供されており、こちらをつかって学習・評価を行うことができます。

## 学習に必要なもの
* データセット（先程フォルダ分けしてつくった画像データ群）
* データ設定ファイル（先程確認した、wood.yaml）
* モデル設定ファイル（先程確認した、patchcore.yaml）

## 学習Pythonコード
次のセルが学習のPythonコードとなります。

以下で、データ設定ファイルとモデル設定ファイルを指定しています。
```py
model_cfg_path = "patchcore.yaml"
data_cfg_path = "wood.yaml"
output_dir = "outputs"
```
ouutput_dirは学習の結果出来上がったモデルファイルの保存先となります。
output_dirはこのあと、Engineクラスのインスタンス生成のところで、引数default_root_dirに渡されます。

```py
engine = Engine(
    default_root_dir=output_dir,
)
```

fit関数で学習を実行します。

```py
engine.fit(model=model, datamodule=data)
```

## 学習を実行してみましょう
次のセルを実行して、学習を実施してみましょう。
今回の実習環境がCPU環境のためGPU環境に比べて時間がかかりますが、しばらくお待ちください。

Train finished.
と表示されたら学習完了です。

In [ ]:
# 学習処理です
# このセルを実行してみましょう

from anomalib.engine import Engine
from anomalib.data import get_datamodule
from anomalib.models import get_model

# ↓↓↓ jupyter固有の処理
# Anomalibにおいて進捗バーがうまく動作しない不具合の対策
import anomalib.models.components.sampling.k_center_greedy as kcg
kcg.tqdm = lambda iterable, **kwargs: iterable
# ↑↑↑

from omegaconf import OmegaConf
from pathlib import Path

model_cfg_path = "patchcore.yaml"
data_cfg_path = "wood.yaml"
output_dir = "outputs"

model_cfg = OmegaConf.load(model_cfg_path)
data_cfg = OmegaConf.load(data_cfg_path)

data= get_datamodule(data_cfg.data)
model = get_model(model_cfg.model) 

engine = Engine(
    default_root_dir=output_dir,
)

engine.fit(model=model, datamodule=data)

print("Train finished.")

## 学習の結果生成されるファイルについて
学習の結果は先程output_dir変数で指定した、"outputs"というディレクトリに保存されるといいました。

画面左のファイルブラウザーをみると、outputsディレクトリができているのがわかります。

![img](./img/cap_outputs.png)

ファイルブラウザーでoutputsディレクトリの中に入っていくと意外と層が深いです。

```
outputs > Patchcore > wood
```

保存先のルートであるoutputsの下にモデル名であるPatchcore、さらにその下にデータセット名であるwoodの名前を関したディレクトリがあります。
その下にv0、latestといったディレクトリがあります。結果はそれらv0、latestといったディレクトリに格納されます。

latestディレクトリの中身をみてみましょう。
weightsというディレクトリがありその下にlightningというディレクトリがあります。こちらに学習結果であるモデルファイルが保存されています。
```
outputs/
  Patchcore/
    wood/
      latest/
        weights/
          lightning/
            model.ckpt
```

lightningディレクトリの下にあるmodel.ckptというファイルがモデルファイルになります。100MB強ほどのファイルサイズになっています。

> lightningはAnomalib内部で使われているPyTorchをつかった機械学習コードを簡潔に記述するためのフレームワークのことです。


### latestディレクトリとバージョンについて
実はlatestとv0ディレクトリは全く同じもので、latestはv0のエイリアスです。
このあと同じ条件（モデル：PatchCore、データセット：woodを使った学習）で学習するとv0の次にv1、次にv2とバージョンを関したディレクトリができていきます。
latestはつねに最新のバージョンのディレクトリを指すエイリアスとなります。



# PatchCoreの学習内容について
PatchCoreについて複数してみましょう。

PatchCoreの学習のおおまかな流れは以下となります。

* WideResNet50やResNet50といったImageNet学習済モデルを使って、訓練データとなる正常画像のCNN特徴量を抽出します
* これを全訓練データに対して行い、この正常訓練データの特徴量の集合をメモリバンクとします
* そのままだとメモリバンクのサイズは非常に大きく、ストレージ容量や処理時間に影響がでるため、精度を落とさないように間引きしてサイズを削減します（コアセットサンプリング）
* バリデーションデータの正常データと異常データをつかって、モデルが出力する最終スコアがちょうど0.5で分離するような正規化パラメータを求めます

PatchCoreの学習において、一番時間がかかるのがコアセットサンプリングとなります。
（特徴量抽出はCPUでも意外と時間がかかりません）

## 正規化とligtningのフレームワークにおける学習について
Anomalibは内部的にはlightningのフレームワークで動作しています。
lightningの学習処理では、train、validationというフェーズがありますがAnomalibでは手法に応じて手法の各工程をtrain、validationに当てはめています。

PatchCoreはニューラルネットワークの学習がないため、少し特殊な構成になります。
trainフェーズでは特徴抽出やコアセットサンプリングが行われ、「バリデーションデータの正常データと異常データをつかって、モデルが出力する最終スコアがちょうど0.5で分離するような正規化パラメータを求めます」がvalidationフェーズで行われます。
これはvalidationデータをつかって、正常データが0.5以下、異常データが0.5以上に、そして値範囲が0.0～1.0になるように正規化する処理です。min-max正規化で行われ、モデルパラメータとして、しきい値、最大値、最小値が決定します。推論時は最終段で、これらのパラメータをつかってmin-max正規化が行われます。


# 学習のハイパーパラメータについて
PatchCoreでは以下３つのハイパーパラメータがあります。

* 特徴量抽出に使うモデル（backbone）
* コアセットサンプリング比率（coreset_sampling_ratio）
* 異常スコア算出時に使う近傍の数 (num_neighbors)

## 特徴量抽出に使うモデル（backbone）
特徴量抽出に使う学習済CNNモデルを選択します。

主に以下の3つがよく使われます。
カッコの中はpatchcore.yamlのbackboneに記述する値です。
デフォルトはwide_resnet50_2です。

* WideResNet50（wide_resnet50_2）
* ResNet50（resnet50）
* ResNet18（resnet18）

モデルは性能とモデル規模のトレードオフとなります。
モデル規模は

```
WideResNet50 > ResNet50 > ResNet18
```
となりますが、モデル規模が大きくなるほど特徴を捉える表現力が上がり、最終的な異常検知性能が向上する傾向があります。
ただモデル規模が大きくなるとメモリ使用量が増え、処理速度が遅くなります。
まずWideResNet50を試して、その環境や業務の要件に応じて、処理速度・検出性能を考慮して決めるのがよいです。

## コアセットサンプリング比率（coreset_sampling_ratio）
コアセットサンプルリングで元のメモリバンクからどれくらいの比率までデータ削減するかを指定します。
たとえば0.1の場合、90%のデータを削減します（10%のデータを残します）。
デフォルトは0.1です。

削減する比率と異常検知性能のトレードオフとなりますが、コアセットサンプリングのアルゴリズムが優秀なため意外と多くのデータを削減しても性能が維持できます。

推論時はメモリバンクの正常特徴と検査特徴の距離を総当りで計算するため、メモリバンクのサイズが大きくなるほど処理時間は長くなります。
またメモリバンクのサイズが増えるためストレージ使用量も上がります。
もし推論速度を向上したい、モデルファイルサイズを削減したいなど必要がでたら、一度コアセットサンプリング比率を下げて、性能が落ちないかどうかを確認するのが良いです。

## 異常スコア算出時に使う近傍の数 (num_neighbors)
異常スコアは画像全体に対する異常度を示すスコアで、異常マップは各局所領域（パッチ）ごとの異常スコアになります。
異常スコアは異常マップの最大スコア(s\*)のものをベースとします。
最大スコアのパッチに対応したメモリ特徴（m*）に関して、たまたま正常データの中の珍しいものにあたったのか、そうでないのかを区別するための補正となります。
m\*に近い上位k個のメモリマップ特徴（k個にはm\*自身も含む）のなかでm*がレア度に応じてs*に補正をかけます。

* m\*のレア度が高い＝たまたま珍しい正常特徴にあたったので異常スコアが高くなった　→　s\*が低くなるよう重み付け
* m\*のレア度が低い= あくまで普通のありふれた正常特徴なので、どの正常よりも遠いので本当に異常といえる　→　s\*が高くなるよう重み付け

このk個が本パラメータとなり、デフォルトは9です。

![img](./img/cap_num_k.png)

#### 値を小さくする
* 異常に敏感になり小さな以上でもスコアが上がりやすい
* ノイズに弱い、メモリバンクの偏りに影響されるなどのデメリット

#### 値を大きくする
* スコア安定し誤検出が減る
* 小さい異常を見逃すことがあるなどのデメリット

<hr>

# モデル評価（PatchCore）
学習して作成したPatchCoreモデルをテストデータで評価します。
評価にはtestメソッドを使います。

## 精度指標
テスト結果として以下４つの精度指標が表示されます。
* image_AUROC
* image_F1Score
* pixel_AUROC
* pixel_F1Score

こちらは座学研修のほうで出てきた内容ですね。
image_xxxxは画像全体に対する評価結果で、pixel_xxxxは１ピクセルごとの評価結果となります。

![img](./img/cap_test_result.png)

### ピクセルごとの評価について
ピクセルごとの評価を実施するにはマスク画像が必要となります。
（データYAMLのmask_dirで指定したデータです）
マスク画像が指定されていない場合、ピクセルごとの評価は行われずに、image_AUROCとimage_F1Scoreの２つだけが返されます。

### 実行してみましょう
次のセルを実行して評価を行ってみましょう。

In [ ]:
# テスト処理です
# このセルを実行してみましょう
engine.test(model=model, datamodule=data)

## テスト結果ファイルの保存先について
テスト結果としてテスト結果と異常マップを合成した画像データが保存されます。

学習のときのwightsと同じ階層にimagesディレクトリが生成され、その下にテスト画像に対する評価結果の可視化画像が保存される。
```
outputs/
  Patchcore/
    wood/
      latest/
        weights/
          lightning/
            model.ckpt
        images/
          test/
            NG/
              NGテスト画像の結果可視化画像 ★
            OK/
              OKテスト画像の結果可視化画像 ★
```

<hr>

# テスト結果画像を見てみましょう
テスト結果としてテストデータに異常マップを合成した画像データも生成されます。


## 異常画像の結果を可視化

In [ ]:
# 異常画像の結果を可視化
# 実行してみましょう

paths = get_image_path_list("outputs/Patchcore/wood/latest/images/test/NG", pathlib=False)

show_images(paths[:6])

## 正常画像の結果を可視化

In [ ]:
# 異常画像の結果を可視化
# 実行してみましょう

paths = get_image_path_list("outputs/Patchcore/wood/latest/images/test/OK", pathlib=False)

show_images(paths[:6])